In [1]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import plotly.graph_objects as go
import trimesh
from build123d import (
    BuildSketch,
    Circle,
    Location,
    Locations,
    Mode,
    Plane,
    Rectangle,
    export_step,
    export_stl,
    extrude,
    import_step,
)

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
DESKTOP = Path.home() / "Desktop"
JSON_PATH = Path("/Users/softage/Desktop/Thelio/PN-000666_v7_export.json")
STEP_PATH = Path("/Users/softage/Desktop/Thelio/PN-000666 v7.step")

OUT_STL = DESKTOP / "PN-000666 v7.stl"
OUT_STEP = DESKTOP / "PN-000666 v7.step"

# -----------------------------------------------------------------------------
# Load Fusion parameters
# Fusion export values here are stored in cm-style numbers, so convert to mm.
# Example: 0.179 -> 1.79 mm
# -----------------------------------------------------------------------------
with open(JSON_PATH) as f:
    data = json.load(f)

P = {p["name"]: p["value"] for p in data.get("parameters", [])}


def mm(name: str) -> float:
    return P[name] * 10.0


# -----------------------------------------------------------------------------
# Parametric dimensions from JSON
# -----------------------------------------------------------------------------
THICKNESS = mm("d10")          # 1.79 mm, symmetric extrude
CENTER_R = mm("d9")            # 6.45 mm
END_CENTER = mm("d4") / 2      # 10.0 mm
END_R = mm("d2")               # 3.5 mm
HOLE_R = mm("d6") / 2          # 1.6 mm

# If you want the profile to match the STEP volume even more closely,
# you can use 6.52 instead of CENTER_R.
# CENTER_R = 6.52


# -----------------------------------------------------------------------------
# Model
# Sketch lies on the YZ plane, then extrudes symmetrically along X.
# -----------------------------------------------------------------------------
def build_part():
    with BuildSketch(Plane.YZ) as sketch:
        # Main outer shape: center disk + horizontal capsule
        Circle(CENTER_R)
        with Locations(Location((0, END_CENTER, 0)), Location((0, -END_CENTER, 0))):
            Circle(END_R)
        Rectangle(2 * END_R, 2 * END_CENTER)

        # Side through-holes
        with Locations(Location((0, END_CENTER, 0)), Location((0, -END_CENTER, 0))):
            Circle(HOLE_R, mode=Mode.SUBTRACT)

    return extrude(sketch.sketch, amount=THICKNESS / 2, both=True)


def export_part(part):
    export_stl(part, OUT_STL)
    export_step(part, OUT_STEP)
    return OUT_STL, OUT_STEP


def plotly_figure_from_part(part=None):
    if part is None:
        part = build_part()

    with tempfile.NamedTemporaryFile(suffix=".stl", delete=False) as tmp:
        temp_path = Path(tmp.name)

    export_stl(part, temp_path)
    mesh = trimesh.load_mesh(str(temp_path), force="mesh")
    temp_path.unlink(missing_ok=True)

    v = mesh.vertices
    f = mesh.faces

    fig = go.Figure(
        data=[
            go.Mesh3d(
                x=v[:, 0],
                y=v[:, 1],
                z=v[:, 2],
                i=f[:, 0],
                j=f[:, 1],
                k=f[:, 2],
                color="lightsteelblue",
                opacity=1.0,
                flatshading=False,
                lighting=dict(
                    ambient=0.4,
                    diffuse=0.9,
                    specular=0.6,
                    roughness=0.4,
                    fresnel=0.2,
                ),
                lightposition=dict(x=100, y=200, z=150),
            )
        ]
    )

    fig.update_layout(
        title="PN-000666 v7 - Parametric Build123d",
        scene_aspectmode="data",
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
        ),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig


def validate_against_step(part):
    step_ref = import_step(STEP_PATH)
    bb = part.bounding_box()
    ref_bb = step_ref.bounding_box()

    print("JSON:", JSON_PATH)
    print("STEP reference:", STEP_PATH)
    print("Part volume:", round(part.volume, 6), "mm^3")
    print("STEP volume:", round(step_ref.volume, 6), "mm^3")
    print(
        "Volume error:",
        round(abs(part.volume - step_ref.volume), 6),
        "mm^3",
    )
    print("Part bbox min:", tuple(round(v, 3) for v in bb.min))
    print("Part bbox max:", tuple(round(v, 3) for v in bb.max))
    print("STEP bbox min:", tuple(round(v, 3) for v in ref_bb.min))
    print("STEP bbox max:", tuple(round(v, 3) for v in ref_bb.max))


# -----------------------------------------------------------------------------
# Run
# -----------------------------------------------------------------------------
part = build_part()
stl_path, step_path = export_part(part)
validate_against_step(part)

print("STL exported to:", stl_path)
print("STEP exported to:", step_path)

fig = plotly_figure_from_part(part)
fig.show(renderer="notebook_connected")


JSON: /Users/softage/Desktop/Thelio/PN-000666_v7_export.json
STEP reference: /Users/softage/Desktop/Thelio/PN-000666 v7.step
Part volume: 371.333823 mm^3
STEP volume: 374.98543 mm^3
Volume error: 3.651607 mm^3
Part bbox min: (-0.895, -6.45, -13.5)
Part bbox max: (0.895, 6.45, 13.5)
STEP bbox min: (0.0, -13.5, -6.5)
STEP bbox max: (1.79, 13.5, 6.5)
STL exported to: /Users/softage/Desktop/PN-000666 v7.stl
STEP exported to: /Users/softage/Desktop/PN-000666 v7.step
